In [138]:
import pandas as pd

In [139]:
df = pd.DataFrame({
    "participant_id": [1,1,1,2,2],
    "timestamp": [
        "2026-01-03 09:15:32.120",
        "2026-01-03 09:15:35.842",
        "2026-01-03 09:15:39.001",
        "2026-01-04 10:02:11.450",
        "2026-01-04 10:02:15.900"
    ],
    "rt": [520, 610, 580, 490, 530],
    "accuracy": [1,1,0,1,1]
})

> pd.to_datetime() converts heterogeneous string or numeric representations of time into pandas’ datetime64[ns] dtype.

Key parameters:
- format= : explicitly specify format to avoid ambiguous parsing and improve speed.
- errors= : "raise", "coerce", or "ignore".
- utc= : convert to UTC.
- dayfirst= : resolve locale ambiguity.

In [140]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%Y-%m-%d %H:%M:%S.%f"
)

df["iti"] = df["timestamp"].diff().dt.total_seconds() # .dt accessor exposes datetime-specific vectorized operations.

print(df)

   participant_id               timestamp   rt  accuracy        iti
0               1 2026-01-03 09:15:32.120  520         1        NaN
1               1 2026-01-03 09:15:35.842  610         1      3.722
2               1 2026-01-03 09:15:39.001  580         0      3.159
3               2 2026-01-04 10:02:11.450  490         1  89192.449
4               2 2026-01-04 10:02:15.900  530         1      4.450


##### Indexing with dates

When time is the primary axis of analysis, it should be the index. A DatetimeIndex allows:
- Partial string indexing (e.g., "2026-01-03").
- Slicing by date ranges.
- Time-aware resampling.

In [141]:
# Set index:
df = df.set_index("timestamp")
print(df.index.name)
print(df.loc["2026-01-03"])

timestamp
                         participant_id   rt  accuracy    iti
timestamp                                                    
2026-01-03 09:15:32.120               1  520         1    NaN
2026-01-03 09:15:35.842               1  610         1  3.722
2026-01-03 09:15:39.001               1  580         0  3.159


##### Basic resampling
Resampling aggregates time-indexed data to a different temporal resolution.

In [142]:
# Daily average RT
daily_mean_rt = df["rt"].resample("D").mean() # D as daily (H as hourly, T or min as minute, W as weekly
print(daily_mean_rt)

timestamp
2026-01-03    570.0
2026-01-04    510.0
Freq: D, Name: rt, dtype: float64


Note: Always verify that the resampling window corresponds to a theoretically justified level (for example, day as independent session vs arbitrary calendar boundary).

In [143]:
# Let's create a trial timestamp column from a session_date column and a trial_time column.

df_new = pd.DataFrame({
    "participant_id": [1,1,1,2,2],
    "session_date": [
        "2026-01-03",
        "2026-01-03",
        "2026-01-03",
        "2026-01-04",
        "2026-01-04"
    ],
    "trial_time": [
        "09:15:32.120",
        "09:15:35.842",
        "09:15:39.001",
        "10:02:11.450",
        "10:02:15.900"
    ],
    "rt": [520, 610, 580, 490, 530],
    "accuracy": [1,1,0,1,1]
})

print(df_new)

# Combine into full timestamp.
df_new["trial_timestamp"] = pd.to_datetime(
    df_new["session_date"] + " " + df_new["trial_time"],
    format="%Y-%m-%d %H:%M:%S.%f"
)

print(df_new)

   participant_id session_date    trial_time   rt  accuracy
0               1   2026-01-03  09:15:32.120  520         1
1               1   2026-01-03  09:15:35.842  610         1
2               1   2026-01-03  09:15:39.001  580         0
3               2   2026-01-04  10:02:11.450  490         1
4               2   2026-01-04  10:02:15.900  530         1
   participant_id session_date    trial_time   rt  accuracy  \
0               1   2026-01-03  09:15:32.120  520         1   
1               1   2026-01-03  09:15:35.842  610         1   
2               1   2026-01-03  09:15:39.001  580         0   
3               2   2026-01-04  10:02:11.450  490         1   
4               2   2026-01-04  10:02:15.900  530         1   

          trial_timestamp  
0 2026-01-03 09:15:32.120  
1 2026-01-03 09:15:35.842  
2 2026-01-03 09:15:39.001  
3 2026-01-04 10:02:11.450  
4 2026-01-04 10:02:15.900  


If trial_time is milliseconds since session start:
> df_new["session_start"] = pd.to_datetime(df_new["session_date"])   
> df_new["trial_timestamp"] = (df_new["session_start"] + pd.to_timedelta(df_new["trial_time"], unit="ms"))

This is common in cognitive experiments where logs store relative timing.

In [144]:
# Let's filter trials by date range.

# First, setting index:
df_new = df_new.set_index("trial_timestamp")

# Second, filter:
subset = df["2026-01-03":"2026-01-04"]

print(subset)

                         participant_id   rt  accuracy        iti
timestamp                                                        
2026-01-03 09:15:32.120               1  520         1        NaN
2026-01-03 09:15:35.842               1  610         1      3.722
2026-01-03 09:15:39.001               1  580         0      3.159
2026-01-04 10:02:11.450               2  490         1  89192.449
2026-01-04 10:02:15.900               2  530         1      4.450


Common Mistakes:
- Time zones: if experiments span labs or online participants, enforce UTC during conversion.
- Missing timestamps: use errors="coerce" and inspect NaT values.
- Non-monotonic index: resampling requires sorted index:
> df = df.sort_index()
- Aggregation bias: daily means assume independence at day level; if trials cluster unevenly across days, weighting issues arise.